# Orquestração Reativa com Árvores de Comportamento (*Behavior Trees*)
## Estação de Reabastecimento de Hidrogênio (HRS) • SCADA-Core (ECAA08)

Este notebook implementa um motor completo de **Árvore de Comportamento (BT)** orientado a objetos em Python e simula:
1. **Cenário 1:** Ciclo completo nominal de abastecimento SAE J2601 até $700\text{ bar}$;
2. **Cenário 2:** Preempção reativa em tempo real quando ocorre vazamento de $\text{H}_2$ (`AT-101 >= 40% LFL`) no meio do abastecimento.

In [ ]:
from enum import Enum
from typing import List, Callable, Dict, Any
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

class Status(Enum):
    SUCCESS = "SUCCESS"
    FAILURE = "FAILURE"
    RUNNING = "RUNNING"

class BTNode:
    def __init__(self, name: str):
        self.name = name
    def tick(self, bb: Dict[str, Any]) -> Status:
        raise NotImplementedError

class Sequence(BTNode):
    def __init__(self, name: str, children: List[BTNode]):
        super().__init__(name)
        self.children = children
    def tick(self, bb: Dict[str, Any]) -> Status:
        for child in self.children:
            status = child.tick(bb)
            if status != Status.SUCCESS:
                return status
        return Status.SUCCESS

class Fallback(BTNode):
    def __init__(self, name: str, children: List[BTNode]):
        super().__init__(name)
        self.children = children
    def tick(self, bb: Dict[str, Any]) -> Status:
        for child in self.children:
            status = child.tick(bb)
            if status != Status.FAILURE:
                return status
        return Status.FAILURE

class Condition(BTNode):
    def __init__(self, name: str, fn: Callable[[Dict[str, Any]], bool]):
        super().__init__(name)
        self.fn = fn
    def tick(self, bb: Dict[str, Any]) -> Status:
        return Status.SUCCESS if self.fn(bb) else Status.FAILURE

class Action(BTNode):
    def __init__(self, name: str, fn: Callable[[Dict[str, Any]], Status]):
        super().__init__(name)
        self.fn = fn
    def tick(self, bb: Dict[str, Any]) -> Status:
        return self.fn(bb)

### 2. Definição das Condições e Ações da Estação de Hidrogênio (Blackboard SCADA-Core)

In [ ]:
def sis_seguro(bb: Dict[str, Any]) -> bool:
    return (
        bb["ESD_100"] == 0
        and bb["AT_101"] < 40.0
        and bb["PT_101"] < 400.0
        and bb["TT_101"] < 85.0
    )

def bico_acoplado(bb: Dict[str, Any]) -> bool:
    return bb["bico_conectado"] and bb["aterramento_ok"]

def acao_teste_estanqueidade(bb: Dict[str, Any]) -> Status:
    if bb["leak_check_ok"]:
        return Status.SUCCESS
    bb["fase"] = "Teste Estanqueidade"
    bb["leak_check_ticks"] += 1
    if bb["leak_check_ticks"] >= 2:
        bb["leak_check_ok"] = True
        return Status.SUCCESS
    return Status.RUNNING

def acao_resfriar_chiller(bb: Dict[str, Any]) -> Status:
    if bb["T_chiller"] <= -38.0:
        return Status.SUCCESS
    bb["fase"] = "Pre-resfriamento RESF-01"
    bb["T_chiller"] -= 15.0
    return Status.RUNNING

def acao_abastecer_cascata(bb: Dict[str, Any]) -> Status:
    if bb["P_veic"] >= 700.0:
        return Status.SUCCESS
    bb["fase"] = "Abastecimento Cascata (TK-BP/TK-AP)"
    bb["XV_301"] = 1
    bb["P_veic"] = min(700.0, bb["P_veic"] + 65.0)
    return Status.RUNNING

def acao_despressurizar_liberar(bb: Dict[str, Any]) -> Status:
    bb["fase"] = "Concluido / Bico Liberado"
    bb["XV_301"] = 0
    return Status.SUCCESS

def acao_abortar_sis(bb: Dict[str, Any]) -> Status:
    bb["fase"] = "TRIP SIS - Valvulas Fechadas"
    bb["XV_101"] = 0
    bb["XV_301"] = 0
    bb["alarme"] = 1
    return Status.SUCCESS

def construir_bt_hrs() -> BTNode:
    return Fallback("Raiz_HRS", [
        Sequence("Ciclo_Seguro_SAE_J2601", [
            Condition("SIS_OK", sis_seguro),
            Sequence("Pre_Abastecimento", [
                Condition("Bico_e_Aterramento", bico_acoplado),
                Action("Teste_Estanqueidade", acao_teste_estanqueidade),
            ]),
            Sequence("Resfriamento_e_Rampa", [
                Action("Chiller_RESF01", acao_resfriar_chiller),
                Action("Rampa_Cascata_700bar", acao_abastecer_cascata),
            ]),
            Action("Despressurizar_Liberar", acao_despressurizar_liberar),
        ]),
        Action("Trip_Emergencia_SIS", acao_abortar_sis),
    ])

### 3. Simulação: Ciclo Nominal vs. Preempção por Vazamento (`AT-101 >= 40% LFL`)

In [ ]:
def simular_cenario(injetar_vazamento_tick=None):
    bt = construir_bt_hrs()
    bb = {
        "ESD_100": 0, "AT_101": 2.0, "PT_101": 340.0, "TT_101": 28.0,
        "bico_conectado": True, "aterramento_ok": True,
        "leak_check_ok": False, "leak_check_ticks": 0,
        "T_chiller": 10.0, "P_veic": 50.0,
        "XV_101": 1, "XV_301": 0, "alarme": 0, "fase": "Inicio"
    }
    historico = []
    for t in range(1, 20):
        if injetar_vazamento_tick is not None and t >= injetar_vazamento_tick:
            bb["AT_101"] = 55.0  # Vazamento critico H2 >= 40% LFL
        status = bt.tick(bb)
        historico.append({
            "Tick": t, "Status_Raiz": status.value, "Fase": bb["fase"],
            "P_veic (bar)": bb["P_veic"], "T_chiller (C)": bb["T_chiller"],
            "AT_101 (%LFL)": bb["AT_101"], "XV_301": bb["XV_301"], "Alarme": bb["alarme"]
        })
        if status == Status.SUCCESS:
            break
    return pd.DataFrame(historico)

df_nom = simular_cenario()
df_vaz = simular_cenario(injetar_vazamento_tick=10)

print("--- CENARIO 1: CICLO NOMINAL ---")
display(df_nom)
print("\n--- CENARIO 2: PREEMPCAO POR VAZAMENTO NO TICK 10 ---")
display(df_vaz)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(df_nom["Tick"], df_nom["P_veic (bar)"], marker="o", color="green", label="Nominal (Conclui 700 bar)")
ax.plot(df_vaz["Tick"], df_vaz["P_veic (bar)"], marker="x", color="red", linestyle="--", label="Vazamento AT-101 no Tick 10 (Trip SIS)")
ax.axvline(10, color="orange", linestyle=":", label="Disparo AT-101 >= 40% LFL")
ax.set_title("Evolução da Pressão no Veículo FCEV sob Controle por Behavior Tree")
ax.set_xlabel("Tick SCADA (100 ms/tick simulado)")
ax.set_ylabel("Pressão P_veic [bar]")
ax.legend()
plt.tight_layout()
plt.show()